# 01 — Comprensión de datos

## Proyecto: HR Process Automation Scanner


El objetivo de este notebook es explorar los datasets descargados para decidir cuál será la fuente principal del proyecto.

Este proyecto busca construir un recomendador de automatización de procesos HR basado en datos, Machine Learning y lógica de negocio.

La solución final permitirá:

- analizar procesos de HR Operations;
- calcular un score de oportunidad de automatización;
- clasificar procesos según prioridad;
- recomendar el tipo de solución más adecuada;
- estimar impacto operativo;
- construir una aplicación en Streamlit orientada a la toma de decisiones.

## Diferencia importante:

No todos los procesos deben automatizarse con IA. Este proyecto recomendará dónde tiene sentido aplicar IA y dónde no.

Algunos procesos sí son buenos candidatos:

repetitivos;
frecuentes;
basados en reglas;
con bajo riesgo;
con datos disponibles;
con mucha carga manual.

Otros no conviene automatizarlos totalmente:

casos legales complejos;
employee relations sensibles;
conflictos laborales;
decisiones disciplinarias;
situaciones con alta carga humana o ética.

## El producto final

El resultado será una app en Streamlit tipo:

HR Process Automation Scanner

Donde el usuario podrá:

Ver ranking de procesos HR.
Identificar quick wins de automatización.
Simular un nuevo proceso.
Recibir una recomendación automática.
Ver qué tipo de tecnología aplicar.
Estimar impacto operativo.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data"
RAW_DATA_PATH = DATA_PATH / "raw"
PROCESSED_DATA_PATH = DATA_PATH / "processed"
FINAL_DATA_PATH = DATA_PATH / "final"

print("Carpeta actual:", Path.cwd())
print("Raíz del proyecto:", PROJECT_ROOT)
print("Ruta raw:", RAW_DATA_PATH)
print("Existe data/raw:", RAW_DATA_PATH.exists())

In [ ]:
for path in RAW_DATA_PATH.rglob("*"):
    if path.is_file():
        print(path)

In [ ]:
# Detectar y cargar los CSV
csv_files = list(RAW_DATA_PATH.rglob("*.csv"))

csv_files

In [ ]:
# Crear función de inspección
def inspect_dataset(file_path, nrows=5):
    """
    Carga e inspecciona un dataset CSV.

    Parámetros
    ----------
    file_path : str or Path
        Ruta del archivo CSV.
    nrows : int
        Número de filas a mostrar como vista previa.

    Devuelve
    --------
    pd.DataFrame
        DataFrame cargado.
    """
    file_path = Path(file_path)

    print("=" * 100)
    print(f"ARCHIVO: {file_path.name}")
    print("=" * 100)

    try:
        df = pd.read_csv(file_path)

        print(f"Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas")

        print("\nColumnas:")
        print(df.columns.tolist())

        print("\nTipos de datos:")
        display(df.dtypes.to_frame("tipo_dato"))

        print("\nValores nulos:")
        display(df.isna().sum().to_frame("valores_nulos"))

        print("\nVista previa:")
        display(df.head(nrows))

        return df

    except Exception as error:
        print(f"Error al cargar el archivo: {error}")
        return None

In [ ]:
# Cargar todos los datasets
datasets = {}

for file_path in csv_files:
    dataset_name = file_path.stem.lower().replace(" ", "_").replace("-", "_")
    datasets[dataset_name] = inspect_dataset(file_path)

In [ ]:
# Resumen comparativo:
dataset_summary = []

for name, df in datasets.items():
    if df is not None:
        dataset_summary.append({
            "nombre_dataset": name,
            "filas": df.shape[0],
            "columnas": df.shape[1],
            "nombres_columnas": ", ".join(df.columns.astype(str))
        })

dataset_summary_df = pd.DataFrame(dataset_summary)

dataset_summary_df

In [ ]:
# Ver columnas por dataset
for name, df in datasets.items():
    print("\n" + "=" * 100)
    print(f"DATASET: {name}")
    print("=" * 100)
    print(df.columns.tolist())

## Decisión inicial

Usaremos como dataset principal: **customer_support_tickets_200k**

| Criterio | Evaluación |
|----------|------------|
| Volumen | Muy alto: 200.000 filas |
| Columnas | 30 columnas |
| Valor operativo | Alto |
| Potencial para ML | Alto |
| Potencial para simulación | Alto |
| Potencial para app Streamlit | Alto |

---

## Rol de cada dataset

| Dataset | Filas | Columnas | Uso recomendado |
|---------|-------|----------|-----------------|
| customer_support_tickets_200k | 200.000 | 30 | Dataset principal |
| customer_support_tickets | 8.469 | 17 | Dataset secundario / comparación / backup |
| helpdesk_event_log | 13.710 | 3 | Dataset complementario para lógica de proceso |

---

## Cómo interpretaremos cada dataset

### 1. customer_support_tickets_200k

Este será nuestro equivalente a tickets internos de HR Operations.

| Dataset original | Interpretación en el proyecto |
|------------------|-------------------------------|
| ticket | Ticket interno de HR |
| category | Proceso HR |
| issue_description | Descripción del caso |
| priority | Prioridad del ticket |
| status | Estado del caso |
| channel | Canal de entrada |
| region | Región o mercado |
| resolution_notes | Notas de resolución |
| customer_age / gender | Variables que probablemente eliminaremos |
| subscription_type | Podría transformarse en tipo de empleado o segmento |
| customer_tenure_months | Podría transformarse en antigüedad del empleado |

### 2. customer_support_tickets

Lo usaremos solo si necesitamos comparar estructura o enriquecer ideas. Tiene menos filas y menos columnas, así que no será el principal.

### 3. helpdesk_event_log

Tiene solo tres columnas: `CaseID`, `ActivityID`, `CompleteTimestamp`.

No sirve como dataset principal, pero es útil para calcular lógica de proceso:

| Uso |
|-----|
| Número de actividades por caso |
| Duración total del caso |
| Cantidad de pasos |
| Complejidad del flujo |
| Posible estandarización |
| Variabilidad entre casos |

## Inspeccionar dataset principal:

Exploraremos:

- todas sus columnas completas;

- tipos de datos;

- nulos;

- valores únicos;

- primeras filas;

- si tiene fechas;


si tiene variables útiles para crear:

- volumen mensual;

- tiempo medio;

- tasa de errores;

- tasa de resolución;

- prioridad;

- canal;

- categoría;

- posible score de automatización.

In [ ]:
# Extraer el dataset principal:
df_main = datasets["customer_support_tickets_200k"].copy()

df_main.shape

In [ ]:
# Ver todas las columnas completas:
df_main.columns.tolist()

In [ ]:
# Primeras filas:
df_main.head()

In [ ]:
# Ver tipos de datos:
df_main.info()

In [ ]:
df_main.dtypes.to_frame("tipo_dato")

In [ ]:
# Ver valores nulos:
missing_summary = (
    df_main
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "columna", 0: "valores_nulos"})
)

missing_summary["porcentaje_nulos"] = (
    missing_summary["valores_nulos"] / len(df_main) * 100
).round(2)

missing_summary.sort_values("porcentaje_nulos", ascending=False)

In [ ]:
# Ver variables categóricas:
categorical_columns = df_main.select_dtypes(include="object").columns.tolist()

categorical_columns

In [ ]:
for column in categorical_columns:
    print("\n" + "=" * 80)
    print(f"COLUMNA: {column}")
    print("=" * 80)
    print(df_main[column].value_counts(dropna=False).head(20))

Vemos valores más frecuentes de columnas como:

- category;

- priority;

- status;

- channel;

- region;

- etc.

In [ ]:
# Ver variables numéricas:
numeric_columns = df_main.select_dtypes(include=["int64", "float64"]).columns.tolist()

numeric_columns

In [ ]:
df_main[numeric_columns].describe().T

In [ ]:
# Guardamos un resumen de columnas - Esto permitirá documentar el notebook:
column_summary = pd.DataFrame({
    "columna": df_main.columns,
    "tipo_dato": df_main.dtypes.astype(str).values,
    "valores_nulos": df_main.isna().sum().values,
    "porcentaje_nulos": (df_main.isna().sum().values / len(df_main) * 100).round(2),
    "valores_unicos": df_main.nunique().values
})

column_summary

## Decisión técnica

Este dataset sirve para el proyecto.

Pero ahora debemos transformarlo.

El dataset original es de soporte al cliente. Por eso, lo vamos a convertir en una versión orientada a HR Operations.

Para eso crearemos un nuevo DataFrame llamado:

df_hr

Este será el dataset de trabajo del proyecto.

## Adaptación inicial del dataset al contexto de HR Operations

El dataset original contiene tickets de soporte al cliente. Para este proyecto, se reutilizará como base operativa para simular tickets internos de HR Operations.

La lógica de adaptación será la siguiente:

- cada ticket representará una solicitud interna de HR;
- la categoría original se transformará en un proceso de HR;
- el producto original se interpretará como sistema o plataforma relacionada;
- las variables de prioridad, canal, región, escalación, SLA y tiempo de resolución se conservarán porque son relevantes para operaciones;
- las variables personales del cliente que no aportan valor directo al objetivo del proyecto serán descartadas o transformadas.

El objetivo es construir una base analítica que permita evaluar qué procesos de HR tienen mayor oportunidad de automatización u optimización con IA.

In [ ]:
# Crear una copia del dataset principal para adaptarlo al contexto de HR Operations
df_hr = df_main.copy()

# Renombrar columnas para alinearlas con el caso de negocio
df_hr = df_hr.rename(columns={
    "ticket_id": "case_id",
    "product": "hr_system",
    "category": "hr_process",
    "issue_description": "case_description",
    "priority": "case_priority",
    "status": "case_status",
    "channel": "contact_channel",
    "customer_tenure_months": "employee_tenure_months",
    "previous_tickets": "previous_cases",
    "customer_satisfaction_score": "employee_satisfaction_score",
    "issue_complexity_score": "process_complexity_score",
    "customer_segment": "employee_segment"
})

df_hr.head()

In [ ]:
# Revisar columnas nuevas:
for index, column in enumerate(df_hr.columns, start=1):
    print(f"{index}. {column}")

In [ ]:
# Eliminar columnas que no aportan al proyecto:
columns_to_drop = [
    "customer_name",
    "customer_email",
    "customer_age",
    "customer_gender",
    "operating_system",
    "browser",
    "payment_method"
]

df_hr = df_hr.drop(columns=columns_to_drop)

df_hr.shape

In [ ]:
# Revisar resultado final:
df_hr.head()

In [ ]:
df_hr.info()

# ¿Por qué hacemos esto?

Porque el proyecto no va de clientes, emails, navegadores o métodos de pago.

El proyecto va de:

* procesos HR

* volumen operativo

* tiempos

* SLA

* complejidad

* escalaciones

* prioridad

* automatización


Así que limpiamos la estructura para que el dataset sea un dataset profesional de HR Operations.

## Transformación de procesos:
Hemos conseguido "hr_process" . Pero sus valores todavía vienen del dataset original, por ejemplo:

- Account Suspension
- Performance Issue
- Subscription Cancellation
- Feature Request

Eso todavía no parece HR.

Tenemos que transformarlo en procesos como:

* HRIS Access Issue

* Payroll Support

* Employee Data Update

* Benefits Request

* Onboarding Documentation

* Leave of Absence

* Contract Management

* Internal Transfer

* Training Request

* Compliance Documentation


In [ ]:
# Ver valores únicos actuales de hr_process:
df_hr["hr_process"].value_counts()

Aquí podemos visualizar qué categorías originales existen y cuántos casos tiene cada una.

In [ ]:
# Ver valores únicos de hr_system:
df_hr["hr_system"].value_counts()

Esta información representa el sistema o plataforma relacionada con el caso.

In [ ]:
# Ver valores únicos de variables operativas clave:
columns_to_review = [
    "hr_process",
    "hr_system",
    "case_priority",
    "case_status",
    "contact_channel",
    "region",
    "subscription_type",
    "escalated",
    "sla_breached",
    "language",
    "preferred_contact_time",
    "employee_segment"
]

for column in columns_to_review:
    print("\n" + "=" * 80)
    print(f"COLUMNA: {column}")
    print("=" * 80)
    print(df_hr[column].value_counts(dropna=False))

## Hasta aquí vemos que:

hr_process tiene 10 categorías originales:

1. Feature Request
2. Subscription Cancellation
3. Performance Issue
4. Security Concern
5. Login Issue
6. Payment Problem
7. Bug Report
8. Refund Request
9. Data Sync Issue
10. Account Suspension

hr_system tiene 10 sistemas originales:

1. Billing System
2. CRM Platform
3. E-commerce Store
4. Cloud Storage
5. Mobile App
6. Analytics Dashboard
7. Web Portal
8. Payment Gateway
9. Subscription Service
10. API Service

Ahora vamos a hacer la adaptación profesional a HR.

## 1.5 Adaptación de categorías originales a procesos de HR Operations

El dataset original pertenece a un contexto de soporte al cliente. Para alinear el proyecto con el objetivo de People Analytics y HR Operations, se transformarán las categorías originales en procesos equivalentes de Recursos Humanos.

Esta adaptación no modifica la estructura operativa del dataset: se conservan variables como prioridad, estado, canal, región, escalación, SLA, tiempo de respuesta y tiempo de resolución.

El objetivo es construir una capa de negocio que permita interpretar cada ticket como una solicitud interna de HR Operations.

In [ ]:
# Mapear procesos originales a procesos HR:
# Crear mapeo de categorías originales a procesos reales de HR Operations
hr_process_mapping = {
    "Feature Request": "HR System Enhancement Request",
    "Subscription Cancellation": "Benefits Cancellation Request",
    "Performance Issue": "HR System Performance Issue",
    "Security Concern": "Compliance & Access Security Review",
    "Login Issue": "HRIS Login Issue",
    "Payment Problem": "Payroll Support Request",
    "Bug Report": "HR Platform Bug Report",
    "Refund Request": "Benefits Reimbursement Request",
    "Data Sync Issue": "Employee Data Update",
    "Account Suspension": "HRIS Access Suspension"
}

# Crear nueva columna con el proceso HR adaptado
df_hr["hr_process_name"] = df_hr["hr_process"].map(hr_process_mapping)

# Revisar resultado
df_hr[["hr_process", "hr_process_name"]].drop_duplicates().sort_values("hr_process")


In [ ]:
# Crear mapeo de sistemas originales a sistemas/plataformas de HR
hr_system_mapping = {
    "Billing System": "Payroll System",
    "CRM Platform": "HR Case Management Platform",
    "E-commerce Store": "Benefits Portal",
    "Cloud Storage": "HR Document Management System",
    "Mobile App": "Employee Mobile App",
    "Analytics Dashboard": "People Analytics Dashboard",
    "Web Portal": "Employee Self-Service Portal",
    "Payment Gateway": "Payroll Payment Gateway",
    "Subscription Service": "Benefits Administration System",
    "API Service": "HRIS Integration API"
}

# Crear nueva columna con el sistema HR adaptado
df_hr["hr_system_name"] = df_hr["hr_system"].map(hr_system_mapping)

# Revisar resultado
df_hr[["hr_system", "hr_system_name"]].drop_duplicates().sort_values("hr_system")

In [ ]:
# Verificar si el mapeo dejó valores sin traducir:
# Verificar valores no mapeados
unmapped_processes = df_hr[df_hr["hr_process_name"].isna()]["hr_process"].unique()
unmapped_systems = df_hr[df_hr["hr_system_name"].isna()]["hr_system"].unique()

print("Procesos sin mapear:")
print(unmapped_processes)

print("\nSistemas sin mapear:")
print(unmapped_systems)

In [ ]:
# Revisar distribución de procesos HR adaptados:
df_hr["hr_process_name"].value_counts()

In [ ]:
# Revisar distribución de sistemas HR adaptados:
df_hr["hr_system_name"].value_counts()

In [ ]:
# Tabla resumen de procesos HR:
hr_process_summary = (
    df_hr
    .groupby("hr_process_name")
    .agg(
        total_cases=("case_id", "count"),
        avg_resolution_time_hours=("resolution_time_hours", "mean"),
        avg_first_response_time_hours=("first_response_time_hours", "mean"),
        avg_complexity_score=("process_complexity_score", "mean"),
        avg_satisfaction_score=("employee_satisfaction_score", "mean")
    )
    .reset_index()
)

hr_process_summary = hr_process_summary.sort_values("total_cases", ascending=False)

hr_process_summary

## Proceso hasta aquí:

Ahora el dataset empieza a tener sentido para el proyecto:

| Antes | Ahora |
|-------|-------|
| category | hr_process_name |
| product | hr_system_name |
| customer support | HR Operations case management |
| customer satisfaction | employee satisfaction |
| ticket | HR case |

## Problema técnico importante

Ahora mismo, si agrupamos solo por hr_process_name, tenemos únicamente 10 procesos.

No sirve bien para Machine Learning, es decir, para construir un recomendador fuerte, ya que 10 filas son muy pocas para entrenar un modelo.

Por eso haremos algo más profesional:

Crearemos unidades operativas de proceso


## Creación de unidades operativas de proceso

Para que el análisis sea útil desde una perspectiva operativa, no se analizarán únicamente los procesos generales de HR.

En su lugar, se crearán unidades operativas de proceso combinando:

- proceso HR;
- sistema HR;
- región;
- canal de contacto;
- prioridad del caso.

Esto permite evaluar oportunidades de automatización de forma más granular.

Por ejemplo, no es lo mismo analizar "Payroll Support Request" de forma general que analizar "Payroll Support Request en Europa, recibido por email, con prioridad alta".

Analizaremos combinaciones como:

Payroll Support Request + Payroll System + Europe + Email
Payroll Support Request + Payroll System + Asia + Chat
HRIS Login Issue + Employee Self-Service Portal + Europe + Email
Benefits Reimbursement Request + Benefits Portal + North America + Chat

A eso lo llamaremos:

process_unit_id

Así pasamos de 10 procesos generales a muchas unidades operativas analizables.

Esto tiene mucho más sentido ya que, una empresa, no automatiza solo “Payroll” en abstracto. Automatiza partes concretas del proceso según sistema, región, canal, volumen y complejidad.

Esta granularidad permite construir un recomendador más realista y más útil para la toma de decisiones.

In [ ]:
# Convertir fechas a formato datetime:
df_hr["ticket_created_date"] = pd.to_datetime(df_hr["ticket_created_date"], errors="coerce")
df_hr["ticket_resolved_date"] = pd.to_datetime(df_hr["ticket_resolved_date"], errors="coerce")

df_hr[["ticket_created_date", "ticket_resolved_date"]].head()


In [ ]:
# Crear variables temporales a partir de la fecha de creación del ticket
df_hr["created_year"] = df_hr["ticket_created_date"].dt.year
df_hr["created_month"] = df_hr["ticket_created_date"].dt.month
df_hr["created_dayofweek"] = df_hr["ticket_created_date"].dt.dayofweek
df_hr["created_quarter"] = df_hr["ticket_created_date"].dt.quarter

df_hr[[
    "ticket_created_date",
    "created_year",
    "created_month",
    "created_dayofweek",
    "created_quarter"
]].head()

In [ ]:
# Revisar valores de escalated y sla_breached:
print("Valores de escalated:")
print(df_hr["escalated"].value_counts(dropna=False))

print("\nValores de sla_breached:")
print(df_hr["sla_breached"].value_counts(dropna=False))

In [ ]:
# Convertir variables de texto a variables binarias
df_hr["escalated_flag"] = (
    df_hr["escalated"]
    .astype(str)
    .str.lower()
    .str.strip()
    .map({
        "yes": 1,
        "no": 0,
        "true": 1,
        "false": 0,
        "1": 1,
        "0": 0
    })
)

df_hr["sla_breached_flag"] = (
    df_hr["sla_breached"]
    .astype(str)
    .str.lower()
    .str.strip()
    .map({
        "yes": 1,
        "no": 0,
        "true": 1,
        "false": 0,
        "1": 1,
        "0": 0
    })
)

df_hr[["escalated", "escalated_flag", "sla_breached", "sla_breached_flag"]].head()

In [ ]:
# Verificar binarios:
df_hr[["escalated_flag", "sla_breached_flag"]].isna().sum()

In [ ]:
# Crear process_unit_id (unidad operativa de proceso):
df_hr["process_unit_id"] = (
    df_hr["hr_process_name"].astype(str) + " | " +
    df_hr["hr_system_name"].astype(str) + " | " +
    df_hr["region"].astype(str) + " | " +
    df_hr["contact_channel"].astype(str) + " | " +
    df_hr["case_priority"].astype(str)
)

df_hr[[
    "hr_process_name",
    "hr_system_name",
    "region",
    "contact_channel",
    "case_priority",
    "process_unit_id"
]].head()

In [ ]:
# Ver cuántas unidades operativas hay:
df_hr["process_unit_id"].nunique()

In [ ]:
# Crear resumen por unidad operativa:
# Crear dataset agregado por unidad operativa
process_unit_summary = (
    df_hr
    .groupby([
        "process_unit_id",
        "hr_process_name",
        "hr_system_name",
        "region",
        "contact_channel",
        "case_priority"
    ])
    .agg(
        total_cases=("case_id", "count"),
        avg_resolution_time_hours=("resolution_time_hours", "mean"),
        avg_first_response_time_hours=("first_response_time_hours", "mean"),
        sla_breach_rate=("sla_breached_flag", "mean"),
        escalation_rate=("escalated_flag", "mean"),
        avg_complexity_score=("process_complexity_score", "mean"),
        avg_satisfaction_score=("employee_satisfaction_score", "mean"),
        avg_previous_cases=("previous_cases", "mean"),
        avg_employee_tenure_months=("employee_tenure_months", "mean")
    )
    .reset_index()
)

process_unit_summary.head()

In [ ]:
# Revisar dimensiones del dataset de unidades operativas:
process_unit_summary.shape

In [ ]:
# Ver unidades con más volumen:
process_unit_summary.sort_values("total_cases", ascending=False).head(20)

## Diagnóstico:

Vemos que process_unit_summary tiene:

12.000 filas
15 columnas

Significa que sí conseguimos generar muchas unidades operativas.

Pero hay un problema: las unidades están demasiado fragmentadas.

Uunidades con más volumen tienen solo:

34 casos
33 casos
32 casos
31 casos
30 casos

Eso es poco para calcular métricas estables como:

sla_breach_rate;
escalation_rate;
avg_complexity_score;
avg_satisfaction_score.

Si bien no se considera un error técnico, sí puede ser un problema analítico.

## Decisión Técnica:

Crear una versión mejorada del dataset agregado.

En vez de agrupar por:

proceso + sistema + región + canal + prioridad

se agrupará por:

proceso + sistema + región + canal

Y la prioridad la convertiremos en una métrica interna:

high_priority_rate
urgent_priority_rate

Consieramos esto más profesional ya que la prioridad no debería fragmentar tanto el dataset, sino ayudar a explicar el riesgo y la oportunidad de automatización.

## Ajuste de granularidad de las unidades operativas

La primera versión de las unidades operativas combinaba proceso, sistema, región, canal y prioridad. Aunque esta aproximación generó muchas unidades, también fragmentó demasiado los datos.

Para construir métricas más estables, se ajustará la granularidad eliminando la prioridad como dimensión de agrupación.

La prioridad se mantendrá como variable agregada mediante tasas, por ejemplo:

- porcentaje de casos de prioridad alta;
- porcentaje de casos urgentes.

La unidad operativa final se definirá como:

proceso HR + sistema HR + región + canal de contacto.

In [ ]:
# Crear variables de prioridad:
# Crear variables binarias de prioridad
df_hr["high_priority_flag"] = (
    df_hr["case_priority"]
    .astype(str)
    .str.lower()
    .str.strip()
    .eq("high")
    .astype(int)
)

df_hr["urgent_priority_flag"] = (
    df_hr["case_priority"]
    .astype(str)
    .str.lower()
    .str.strip()
    .eq("urgent")
    .astype(int)
)

df_hr[["case_priority", "high_priority_flag", "urgent_priority_flag"]].head()

In [ ]:
# Crear nueva unidad operativa sin fragmentar por prioridad
df_hr["process_unit_id_v2"] = (
    df_hr["hr_process_name"].astype(str) + " | " +
    df_hr["hr_system_name"].astype(str) + " | " +
    df_hr["region"].astype(str) + " | " +
    df_hr["contact_channel"].astype(str)
)

df_hr[[
    "hr_process_name",
    "hr_system_name",
    "region",
    "contact_channel",
    "process_unit_id_v2"
]].head()

In [ ]:
# Crear dataset agregado por unidad operativa final
process_unit_summary_v2 = (
    df_hr
    .groupby([
        "process_unit_id_v2",
        "hr_process_name",
        "hr_system_name",
        "region",
        "contact_channel"
    ])
    .agg(
        total_cases=("case_id", "count"),
        avg_resolution_time_hours=("resolution_time_hours", "mean"),
        avg_first_response_time_hours=("first_response_time_hours", "mean"),
        sla_breach_rate=("sla_breached_flag", "mean"),
        escalation_rate=("escalated_flag", "mean"),
        avg_complexity_score=("process_complexity_score", "mean"),
        avg_satisfaction_score=("employee_satisfaction_score", "mean"),
        avg_previous_cases=("previous_cases", "mean"),
        avg_employee_tenure_months=("employee_tenure_months", "mean"),
        high_priority_rate=("high_priority_flag", "mean"),
        urgent_priority_rate=("urgent_priority_flag", "mean")
    )
    .reset_index()
)

process_unit_summary_v2.head()

In [ ]:
# Revisar dimensiones del dataset de unidades operativas:
process_unit_summary_v2.shape

In [ ]:
# Ver unidades con más volumen de casos:
process_unit_summary_v2.sort_values("total_cases", ascending=False).head(20)

In [ ]:
# Comparar granularidad anterior vs nueva:
comparison_granularity = pd.DataFrame({
    "dataset": ["process_unit_summary", "process_unit_summary_v2"],
    "filas": [process_unit_summary.shape[0], process_unit_summary_v2.shape[0]],
    "casos_promedio_por_unidad": [
        process_unit_summary["total_cases"].mean(),
        process_unit_summary_v2["total_cases"].mean()
    ],
    "max_casos_por_unidad": [
        process_unit_summary["total_cases"].max(),
        process_unit_summary_v2["total_cases"].max()
    ],
    "min_casos_por_unidad": [
        process_unit_summary["total_cases"].min(),
        process_unit_summary_v2["total_cases"].min()
    ]
})

comparison_granularity

## Validación de la nueva tabla

| Dataset | Filas | Casos promedio por unidad | Máximo de casos por unidad | Mínimo de casos por unidad |
|---------|-------|--------------------------|---------------------------|---------------------------|
| process_unit_summary | 12.000 | 16,67 | 34 | 3 |
| process_unit_summary_v2 | 3.000 | 66,67 | 94 | 37 |

**`process_unit_summary_v2` es la tabla correcta para continuar** — cada unidad operativa tiene muchos más casos, por lo tanto las métricas son más estables.

## Creación del Automation Opportunity Score

El Automation Opportunity Score mide qué unidades operativas tienen mayor potencial para ser automatizadas u optimizadas con IA.

El score combina variables de volumen, tiempo, fricción operativa, riesgo de SLA, escalación, complejidad y prioridad.

La lógica de negocio es la siguiente:

- más volumen implica mayor oportunidad de impacto;
- mayor tiempo de resolución implica mayor carga operativa;
- mayor tiempo de primera respuesta indica posible cuello de botella;
- mayor tasa de incumplimiento de SLA indica riesgo operativo;
- mayor tasa de escalación indica fricción o complejidad;
- mayor complejidad indica necesidad de optimización;
- mayor proporción de casos urgentes o de prioridad alta indica mayor criticidad.

El objetivo no es automatizar todo, sino priorizar dónde una intervención tecnológica podría generar más valor.

In [ ]:
# Crear copia de trabajo para calcular el score de automatización
df_score = process_unit_summary_v2.copy()

df_score.head()

In [ ]:
# Normalizamos las variables numéricas para calcular el score de automatización - Esta función transforma cada variable a escala 0-1:
def min_max_scale(series):
    """
    Normaliza una serie numérica entre 0 y 1.
    Si todos los valores son iguales, devuelve 0 para evitar división por cero.
    """
    min_value = series.min()
    max_value = series.max()
    
    if max_value == min_value:
        return series * 0
    
    return (series - min_value) / (max_value - min_value)

In [ ]:
# Normalizar variables del score:
# Normalizar variables operativas relevantes
df_score["volume_score"] = min_max_scale(df_score["total_cases"])
df_score["resolution_time_score"] = min_max_scale(df_score["avg_resolution_time_hours"])
df_score["first_response_score"] = min_max_scale(df_score["avg_first_response_time_hours"])
df_score["sla_risk_score"] = min_max_scale(df_score["sla_breach_rate"])
df_score["escalation_score"] = min_max_scale(df_score["escalation_rate"])
df_score["complexity_score"] = min_max_scale(df_score["avg_complexity_score"])
df_score["previous_cases_score"] = min_max_scale(df_score["avg_previous_cases"])
df_score["high_priority_score"] = min_max_scale(df_score["high_priority_rate"])
df_score["urgent_priority_score"] = min_max_scale(df_score["urgent_priority_rate"])

df_score[[
    "volume_score",
    "resolution_time_score",
    "first_response_score",
    "sla_risk_score",
    "escalation_score",
    "complexity_score",
    "previous_cases_score",
    "high_priority_score",
    "urgent_priority_score"
]].head()

## Calcular Automation Opportunity Score

Usaremos pesos de negocio — no todos los factores pesan igual.

| Factor | Peso | Motivo |
|--------|------|--------|
| Volumen | 20% | Automatizar procesos frecuentes genera más impacto |
| Tiempo de resolución | 15% | Procesos lentos consumen más capacidad |
| Incumplimiento SLA | 15% | Riesgo operativo directo |
| Escalación | 10% | Indica fricción y necesidad de intervención |
| Complejidad | 10% | Señala procesos difíciles o cargados manualmente |
| Tiempo de primera respuesta | 10% | Mide cuello de botella inicial |
| Casos previos | 5% | Indica recurrencia o historial operativo |
| Prioridad alta | 7.5% | Indica criticidad |
| Prioridad urgente | 7.5% | Indica criticidad máxima |

In [ ]:
# Calcular Automation Opportunity Score con pesos de negocio
df_score["automation_score"] = (
    df_score["volume_score"] * 0.20 +
    df_score["resolution_time_score"] * 0.15 +
    df_score["sla_risk_score"] * 0.15 +
    df_score["escalation_score"] * 0.10 +
    df_score["complexity_score"] * 0.10 +
    df_score["first_response_score"] * 0.10 +
    df_score["previous_cases_score"] * 0.05 +
    df_score["high_priority_score"] * 0.075 +
    df_score["urgent_priority_score"] * 0.075
) * 100

df_score["automation_score"] = df_score["automation_score"].round(2)

df_score[[
    "process_unit_id_v2",
    "total_cases",
    "automation_score"
]].sort_values("automation_score", ascending=False).head(20)

## Crear prioridad de automatización

Clasificaremos las unidades en tres niveles:

- Alta: score igual o superior a 70.

- Media: score entre 50 y 69.99.

- Baja: score menor de 50.

In [ ]:
# Clasificar prioridad de automatización según el score
df_score["automation_priority"] = pd.cut(
    df_score["automation_score"],
    bins=[-1, 50, 70, 100],
    labels=["Baja", "Media", "Alta"]
)

df_score["automation_priority"].value_counts()

In [ ]:
# Crear recomendación de solución tecnológica basada en el proceso y sistema:
def recommend_automation_solution(row):
    """
    Recomienda un tipo de solución de automatización según las características operativas.
    """
    if row["automation_score"] >= 70:
        if row["avg_complexity_score"] >= 6.5 and row["sla_breach_rate"] >= 0.5:
            return "IA asistida con revisión humana"
        elif row["contact_channel"] in ["Email", "Web Form"] and row["avg_complexity_score"] < 6:
            return "Workflow automation"
        elif row["contact_channel"] in ["Chat", "Social Media"] and row["avg_complexity_score"] < 6:
            return "Chatbot o asistente virtual"
        else:
            return "Automatización prioritaria"
    
    elif row["automation_score"] >= 50:
        if row["avg_complexity_score"] >= 6:
            return "Rediseño del proceso antes de automatizar"
        elif row["sla_breach_rate"] >= 0.5:
            return "Automatización parcial con alertas SLA"
        else:
            return "Automatización parcial"
    
    else:
        if row["avg_complexity_score"] >= 7:
            return "Mantener revisión humana"
        else:
            return "Baja prioridad de automatización"


df_score["recommended_solution"] = df_score.apply(recommend_automation_solution, axis=1)

df_score["recommended_solution"].value_counts()

## Estimar horas ahorrables

Supuesto inicial: si una unidad operativa se automatiza parcialmente, se podría ahorrar un porcentaje del tiempo de resolución según su prioridad de automatización.

| Prioridad | Porcentaje de ahorro estimado |
|-----------|-------------------------------|
| Alta | 35% |
| Media | 20% |
| Baja | 5% |

In [ ]:
# Definir porcentaje estimado de ahorro según prioridad de automatización
savings_rate_mapping = {
    "Alta": 0.35,
    "Media": 0.20,
    "Baja": 0.05
}

# Crear tasa estimada de ahorro
df_score["estimated_savings_rate"] = (
    df_score["automation_priority"]
    .astype(str)
    .map(savings_rate_mapping)
)

# Calcular horas estimadas ahorrables
df_score["estimated_hours_saved"] = (
    df_score["total_cases"] *
    df_score["avg_resolution_time_hours"] *
    df_score["estimated_savings_rate"]
).round(2)

df_score[[
    "process_unit_id_v2",
    "automation_priority",
    "automation_score",
    "estimated_savings_rate",
    "estimated_hours_saved"
]].sort_values("estimated_hours_saved", ascending=False).head(20)

In [ ]:
# Crear ranking final de oportunidades de automatización
automation_ranking = (
    df_score
    .sort_values(
        by=["automation_score", "estimated_hours_saved"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

automation_ranking["ranking"] = automation_ranking.index + 1

automation_ranking[[
    "ranking",
    "hr_process_name",
    "hr_system_name",
    "region",
    "contact_channel",
    "total_cases",
    "automation_score",
    "automation_priority",
    "recommended_solution",
    "estimated_hours_saved"
]].head(20)


## Proceso técnico hasta aqui:
1. Crear df_score
2. Normalizar variables
3. Calcular automation_score
4. Crear automation_priority
5. Crear recommended_solution
6. Crear estimated_savings_rate
7. Crear estimated_hours_saved
8. Crear automation_ranking

## Ajuste de la prioridad de automatización mediante percentiles

El primer cálculo del Automation Opportunity Score generó puntuaciones válidas, pero los umbrales fijos no eran adecuados porque ninguna unidad operativa superaba el umbral de 70 puntos.

Para evitar una clasificación artificial, se utilizarán percentiles.

La lógica será:

- prioridad alta: unidades en el 25% superior del score;
- prioridad media: unidades entre el percentil 25 y 75;
- prioridad baja: unidades en el 25% inferior.

Este enfoque permite priorizar las mejores oportunidades relativas dentro del conjunto de procesos analizados.

In [ ]:
# Revisar distribución del score:
df_score["automation_score"].describe()

In [ ]:
# Calcular percentiles:
low_threshold = df_score["automation_score"].quantile(0.25)
high_threshold = df_score["automation_score"].quantile(0.75)

print("Umbral prioridad baja:", round(low_threshold, 2))
print("Umbral prioridad alta:", round(high_threshold, 2))

In [ ]:
# Reclasificar prioridad de automatización:
def classify_automation_priority(score):
    """
    Clasifica la prioridad de automatización usando percentiles.
    """
    if score >= high_threshold:
        return "Alta"
    elif score >= low_threshold:
        return "Media"
    else:
        return "Baja"


df_score["automation_priority"] = df_score["automation_score"].apply(classify_automation_priority)

df_score["automation_priority"].value_counts()

In [ ]:
# Recalcular tasa de ahorro según nueva clasificación - Ahora que sí tendremos prioridad Alta, recalculamos ahorro:
savings_rate_mapping = {
    "Alta": 0.35,
    "Media": 0.20,
    "Baja": 0.05
}

df_score["estimated_savings_rate"] = (
    df_score["automation_priority"]
    .map(savings_rate_mapping)
)

df_score["estimated_hours_saved"] = (
    df_score["total_cases"] *
    df_score["avg_resolution_time_hours"] *
    df_score["estimated_savings_rate"]
).round(2)

df_score[[
    "process_unit_id_v2",
    "automation_priority",
    "automation_score",
    "estimated_savings_rate",
    "estimated_hours_saved"
]].sort_values("automation_score", ascending=False).head(20)

In [ ]:
# Recalcular recomendación de solución tecnológica con nueva prioridad:
def recommend_automation_solution(row):
    """
    Recomienda un tipo de solución de automatización según prioridad, complejidad,
    canal y riesgo operativo.
    """
    if row["automation_priority"] == "Alta":
        if row["avg_complexity_score"] >= 6.5 and row["sla_breach_rate"] >= 0.5:
            return "IA asistida con revisión humana"
        elif row["contact_channel"] in ["Email", "Web Form"] and row["avg_complexity_score"] < 6:
            return "Workflow automation"
        elif row["contact_channel"] in ["Chat", "Social Media"] and row["avg_complexity_score"] < 6:
            return "Chatbot o asistente virtual"
        elif row["sla_breach_rate"] >= 0.5:
            return "Automatización parcial con alertas SLA"
        else:
            return "Automatización prioritaria"

    elif row["automation_priority"] == "Media":
        if row["avg_complexity_score"] >= 6:
            return "Rediseño del proceso antes de automatizar"
        elif row["sla_breach_rate"] >= 0.5:
            return "Automatización parcial con alertas SLA"
        else:
            return "Automatización parcial"

    else:
        if row["avg_complexity_score"] >= 7:
            return "Mantener revisión humana"
        else:
            return "Baja prioridad de automatización"


df_score["recommended_solution"] = df_score.apply(recommend_automation_solution, axis=1)

df_score["recommended_solution"].value_counts()

In [ ]:
# Rehacer ranking final de oportunidades de automatización con nueva prioridad y recomendaciones:
automation_ranking = (
    df_score
    .sort_values(
        by=["automation_score", "estimated_hours_saved"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

automation_ranking["ranking"] = automation_ranking.index + 1

automation_ranking[[
    "ranking",
    "hr_process_name",
    "hr_system_name",
    "region",
    "contact_channel",
    "total_cases",
    "automation_score",
    "automation_priority",
    "recommended_solution",
    "estimated_hours_saved"
]].head(20)

In [ ]:
# Resumen actualizado: 
executive_summary = pd.DataFrame({
    "métrica": [
        "Unidades operativas analizadas",
        "Score promedio de automatización",
        "Score máximo",
        "Score mínimo",
        "Unidades de prioridad alta",
        "Unidades de prioridad media",
        "Unidades de prioridad baja",
        "Horas estimadas ahorrables totales"
    ],
    "valor": [
        len(df_score),
        round(df_score["automation_score"].mean(), 2),
        round(df_score["automation_score"].max(), 2),
        round(df_score["automation_score"].min(), 2),
        int((df_score["automation_priority"] == "Alta").sum()),
        int((df_score["automation_priority"] == "Media").sum()),
        int((df_score["automation_priority"] == "Baja").sum()),
        round(df_score["estimated_hours_saved"].sum(), 2)
    ]
})

executive_summary


## Diagnóstico

El ranking está bien estructurado:

automation_score funciona.
automation_priority clasifica en Alta / Media / Baja.
recommended_solution genera recomendaciones.
automation_ranking se creó correctamente.

Pero Horas estimadas ahorrables totales no funciona: 5.055.719,93

Es demasiado alto para presentarlo como “horas ahorrables”. 

El problema viene de esta fórmula:

total_cases * avg_resolution_time_hours * estimated_savings_rate

Estamos usando resolution_time_hours, que representa el tiempo total hasta resolver el caso. Pero ese tiempo incluye espera, colas, pausas, dependencia de otros equipos, retrasos y tiempo calendario. No equivale a tiempo manual real de trabajo.

Necesitamos crear una estimación realista:

no ahorramos un porcentaje del tiempo total de resolución, sino del tiempo operativo/manual estimado.

Por este motivo crearemos una variable nueva:

estimated_manual_handling_time_hours



## Ajuste de la estimación de ahorro operativo

La primera estimación de horas ahorrables utilizaba el tiempo total de resolución del caso. Sin embargo, el tiempo de resolución incluye esperas, colas, dependencias externas y tiempo calendario.

Para construir una estimación más realista, se calculará primero un tiempo operativo/manual estimado.

La lógica será:

- los procesos de prioridad alta tendrán mayor proporción de trabajo manual potencialmente optimizable;
- los procesos de prioridad media tendrán una proporción intermedia;
- los procesos de baja prioridad tendrán menor potencial de ahorro.

Esta corrección permite estimar ahorro operativo desde una perspectiva de negocio.

In [ ]:
# Definir proporción estimada de trabajo manual sobre el tiempo total de resolución
manual_work_rate_mapping = {
    "Alta": 0.18,
    "Media": 0.12,
    "Baja": 0.08
}

df_score["estimated_manual_work_rate"] = (
    df_score["automation_priority"]
    .map(manual_work_rate_mapping)
)

df_score[[
    "automation_priority",
    "estimated_manual_work_rate"
]].drop_duplicates()

In [ ]:
# Calcular tiempo manual estimado por caso:
df_score["estimated_manual_handling_time_hours"] = (
    df_score["avg_resolution_time_hours"] *
    df_score["estimated_manual_work_rate"]
).round(2)

df_score[[
    "avg_resolution_time_hours",
    "automation_priority",
    "estimated_manual_work_rate",
    "estimated_manual_handling_time_hours"
]].head()

In [ ]:
# Recalcular horas ahorrables - mantenemos el ahorro estimado por prioridad, pero ahora aplicado sobre el tiempo manual estimado:
# Mantener tasa de ahorro estimado según prioridad
savings_rate_mapping = {
    "Alta": 0.35,
    "Media": 0.20,
    "Baja": 0.05
}

df_score["estimated_savings_rate"] = (
    df_score["automation_priority"]
    .map(savings_rate_mapping)
)

# Calcular horas operativas estimadas ahorrables
df_score["estimated_operational_hours_saved"] = (
    df_score["total_cases"] *
    df_score["estimated_manual_handling_time_hours"] *
    df_score["estimated_savings_rate"]
).round(2)

df_score[[
    "process_unit_id_v2",
    "automation_priority",
    "automation_score",
    "total_cases",
    "avg_resolution_time_hours",
    "estimated_manual_handling_time_hours",
    "estimated_savings_rate",
    "estimated_operational_hours_saved"
]].sort_values("estimated_operational_hours_saved", ascending=False).head(20)


In [ ]:
# Crear ranking final corregido de oportunidades de automatización
automation_ranking = (
    df_score
    .sort_values(
        by=["automation_score", "estimated_operational_hours_saved"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

automation_ranking["ranking"] = automation_ranking.index + 1

automation_ranking[[
    "ranking",
    "hr_process_name",
    "hr_system_name",
    "region",
    "contact_channel",
    "total_cases",
    "automation_score",
    "automation_priority",
    "recommended_solution",
    "estimated_operational_hours_saved"
]].head(20)

In [ ]:
# Resumen corregido:
executive_summary = pd.DataFrame({
    "métrica": [
        "Unidades operativas analizadas",
        "Score promedio de automatización",
        "Score máximo",
        "Score mínimo",
        "Unidades de prioridad alta",
        "Unidades de prioridad media",
        "Unidades de prioridad baja",
        "Horas operativas estimadas ahorrables totales"
    ],
    "valor": [
        len(df_score),
        round(df_score["automation_score"].mean(), 2),
        round(df_score["automation_score"].max(), 2),
        round(df_score["automation_score"].min(), 2),
        int((df_score["automation_priority"] == "Alta").sum()),
        int((df_score["automation_priority"] == "Media").sum()),
        int((df_score["automation_priority"] == "Baja").sum()),
        round(df_score["estimated_operational_hours_saved"].sum(), 2)
    ]
})

executive_summary

## Justificación técnica:

Antes decíamos:

“Si un caso tarda 120 horas en resolverse, automatizarlo ahorra una parte de esas 120 horas.”

No es realista.

Ahora decimos:

“Si un caso tarda 120 horas en resolverse, solo una parte corresponde a trabajo manual estimado. La automatización puede reducir una parte de ese trabajo operativo.”



## Ajuste de la estimación de ahorro operativo

La primera estimación de horas ahorrables utilizaba el tiempo total de resolución del caso. Sin embargo, el tiempo de resolución incluye esperas, colas, dependencias externas y tiempo calendario.

Por este motivo, se ajustará la estimación calculando primero un tiempo operativo/manual estimado.

La lógica será:

- no todo el tiempo de resolución puede ser reducido mediante automatización;
- solo una parte del tiempo corresponde a trabajo operativo o manual;
- la automatización reduce una proporción de ese trabajo manual estimado;
- el ahorro se calcula sobre tiempo operativo estimado, no sobre el tiempo total de resolución.

Con este ajuste logramos, que el impacto estimado, sea más realista desde una perspectiva de negocio.

In [ ]:
# Definir proporción estimada de trabajo manual sobre el tiempo total de resolución
manual_work_rate_mapping = {
    "Alta": 0.18,
    "Media": 0.12,
    "Baja": 0.08
}

df_score["estimated_manual_work_rate"] = (
    df_score["automation_priority"]
    .map(manual_work_rate_mapping)
)

df_score[[
    "automation_priority",
    "estimated_manual_work_rate"
]].drop_duplicates().sort_values("automation_priority")

## Qué significa

| Prioridad | Interpretación |
|-----------|----------------|
| Alta | Se asume que el 18% del tiempo total de resolución corresponde a trabajo manual optimizable |
| Media | Se asume un 12% |
| Baja | Se asume un 8% |

In [ ]:
# Calcular tiempo manual estimado por caso
df_score["estimated_manual_handling_time_hours"] = (
    df_score["avg_resolution_time_hours"] *
    df_score["estimated_manual_work_rate"]
).round(2)

df_score[[
    "hr_process_name",
    "hr_system_name",
    "region",
    "contact_channel",
    "automation_priority",
    "avg_resolution_time_hours",
    "estimated_manual_work_rate",
    "estimated_manual_handling_time_hours"
]].head(20)

In [ ]:
# Definir tasa estimada de ahorro según prioridad de automatización
savings_rate_mapping = {
    "Alta": 0.35,
    "Media": 0.20,
    "Baja": 0.05
}

df_score["estimated_savings_rate"] = (
    df_score["automation_priority"]
    .map(savings_rate_mapping)
)

# Calcular horas operativas estimadas ahorrables
df_score["estimated_operational_hours_saved"] = (
    df_score["total_cases"] *
    df_score["estimated_manual_handling_time_hours"] *
    df_score["estimated_savings_rate"]
).round(2)

df_score[[
    "process_unit_id_v2",
    "automation_priority",
    "automation_score",
    "total_cases",
    "avg_resolution_time_hours",
    "estimated_manual_handling_time_hours",
    "estimated_savings_rate",
    "estimated_operational_hours_saved"
]].sort_values("estimated_operational_hours_saved", ascending=False).head(20)

In [ ]:
# Crear ranking final corregido de oportunidades de automatización
automation_ranking = (
    df_score
    .sort_values(
        by=["automation_score", "estimated_operational_hours_saved"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

automation_ranking["ranking"] = automation_ranking.index + 1

automation_ranking[[
    "ranking",
    "hr_process_name",
    "hr_system_name",
    "region",
    "contact_channel",
    "total_cases",
    "automation_score",
    "automation_priority",
    "recommended_solution",
    "estimated_operational_hours_saved"
]].head(20)

In [ ]:
# Crear resumen ejecutivo corregido
executive_summary = pd.DataFrame({
    "métrica": [
        "Unidades operativas analizadas",
        "Score promedio de automatización",
        "Score máximo",
        "Score mínimo",
        "Unidades de prioridad alta",
        "Unidades de prioridad media",
        "Unidades de prioridad baja",
        "Horas operativas estimadas ahorrables totales"
    ],
    "valor": [
        len(df_score),
        round(df_score["automation_score"].mean(), 2),
        round(df_score["automation_score"].max(), 2),
        round(df_score["automation_score"].min(), 2),
        int((df_score["automation_priority"] == "Alta").sum()),
        int((df_score["automation_priority"] == "Media").sum()),
        int((df_score["automation_priority"] == "Baja").sum()),
        round(df_score["estimated_operational_hours_saved"].sum(), 2)
    ]
})

executive_summary

In [ ]:
# Comparar estimación anterior con estimación corregida
savings_comparison = pd.DataFrame({
    "métrica": [
        "Horas ahorrables estimadas originalmente",
        "Horas operativas ahorrables corregidas",
        "Reducción de la estimación"
    ],
    "valor": [
        round(df_score["estimated_hours_saved"].sum(), 2),
        round(df_score["estimated_operational_hours_saved"].sum(), 2),
        round(
            df_score["estimated_hours_saved"].sum() -
            df_score["estimated_operational_hours_saved"].sum(),
            2
        )
    ]
})

savings_comparison

### Interpretación del ajuste

La estimación inicial de ahorro utilizaba el tiempo total de resolución, lo que podía sobreestimar el impacto real de la automatización.

Para corregirlo, se estimó primero qué parte del tiempo total corresponde a trabajo operativo/manual. Posteriormente, el ahorro potencial se calculó únicamente sobre esa fracción manual estimada.

Este enfoque es más conservador. Reconoce que no todo el tiempo de resolución puede ser reducido mediante automatización. Parte del tiempo corresponde a esperas, dependencias externas, colas de trabajo o tiempos administrativos que no necesariamente desaparecen al automatizar.

## Resumen de resultados

| Métrica | Valor |
|---------|-------|
| Unidades operativas analizadas | 3.000 |
| Score promedio de automatización | 49,44 |
| Score máximo | 66,45 |
| Score mínimo | 31,86 |
| Unidades de prioridad alta | 753 |
| Unidades de prioridad media | 1.497 |
| Unidades de prioridad baja | 750 |
| Horas operativas estimadas ahorrables totales | 739.371,90 |

La distribución tiene sentido:

| Prioridad | Proporción |
|-----------|------------|
| Alta | ~25% |
| Media | ~50% |
| Baja | ~25% |

Eso confirma que el ajuste por percentiles funcionó correctamente.

## Validación del ajuste



| Métrica | Valor |
|---------|-------|
| Horas ahorrables estimadas originalmente | 5.055.719,93 |
| Horas operativas ahorrables corregidas | 739.371,90 |
| Reducción de la estimación | 4.316.348,03 |







> "La automatización reduce una parte estimada del trabajo operativo/manual asociado a cada proceso."

In [ ]:
# Guardar ranking final de oportunidades de automatización
automation_ranking.to_csv(
    FINAL_DATA_PATH / "hr_process_automation_ranking.csv",
    index=False
)

print("Archivo guardado correctamente en:")
print(FINAL_DATA_PATH / "hr_process_automation_ranking.csv")

In [ ]:
# Guardar resumen ejecutivo
executive_summary.to_csv(
    FINAL_DATA_PATH / "executive_summary.csv",
    index=False
)

print("Archivo guardado correctamente en:")
print(FINAL_DATA_PATH / "executive_summary.csv")

In [ ]:
# Guardar comparación de estimaciones de ahorro
savings_comparison.to_csv(
    FINAL_DATA_PATH / "savings_comparison.csv",
    index=False
)

print("Archivo guardado correctamente en:")
print(FINAL_DATA_PATH / "savings_comparison.csv")

In [ ]:
# Verificar guardado de archivos:
for file_path in FINAL_DATA_PATH.glob("*.csv"):
    print(file_path.name)

## Conclusiones del notebook de comprensión de datos

En este notebook se completó la primera fase del proyecto: comprensión, adaptación y preparación inicial de los datos.

A partir de un dataset público de tickets de soporte, se construyó una capa analítica adaptada al contexto de HR Operations. Cada ticket fue interpretado como una solicitud interna de Recursos Humanos y las categorías originales fueron transformadas en procesos y sistemas propios del ecosistema HR.

Durante esta fase se realizaron los siguientes pasos:

- carga e inspección del dataset principal;
- eliminación de columnas no relevantes para el objetivo del proyecto;
- adaptación de categorías originales a procesos de HR Operations;
- adaptación de sistemas originales a plataformas HR;
- creación de unidades operativas de proceso;
- ajuste de granularidad para mejorar la estabilidad de las métricas;
- cálculo del Automation Opportunity Score;
- clasificación de unidades operativas en prioridad alta, media y baja;
- generación de recomendaciones de automatización;
- estimación corregida de horas operativas ahorrables;
- exportación de los datasets finales para las siguientes fases del proyecto.

La tabla final `hr_process_automation_ranking.csv` será utilizada como base para el análisis exploratorio, visualizaciones, modelo de Machine Learning y posterior aplicación en Streamlit.

El resultado de esta fase es un dataset analítico preparado para responder la pregunta principal del proyecto:

¿Qué procesos de HR Operations tienen mayor oportunidad de ser automatizados u optimizados con IA, considerando volumen, tiempos, SLA, escalaciones, complejidad y potencial de ahorro operativo?